In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.preprocessing import LabelEncoder
from google.colab import files
uploaded = files.upload()

Saving uber_trips_dataset_50k.csv to uber_trips_dataset_50k.csv


In [ ]:
df = pd.read_csv("uber_trips_dataset_50k.csv")
df = df[df["status"] == "Completed"].copy()

print("Rows after filtering:", len(df))
df.head()

df["pickup_time"] = pd.to_datetime(df["pickup_time"])

df["hour"]        = df["pickup_time"].dt.hour
df["day_of_week"] = df["pickup_time"].dt.dayofweek   # 0 = Monday, 6 = Sunday
df["is_weekend"]  = df["day_of_week"].isin([5, 6]).astype(int)

def time_bucket(h):
    if   6 <= h < 10: return 0   # Morning rush
    elif 10 <= h < 16: return 1  # Midday
    elif 16 <= h < 20: return 2  # Evening rush
    elif 20 <= h < 24: return 3  # Night
    else:              return 4  # Late night

df["time_bucket"] = df["hour"].apply(time_bucket)

le = LabelEncoder()
df["city_enc"] = le.fit_transform(df["city"])

print("Features added:")
df[["hour", "day_of_week", "is_weekend", "time_bucket", "city_enc"]].head()

Rows after filtering: 42540
Features added:


,hour,day_of_week,is_weekend,time_bucket,city_enc
0,0,6,1,4,4
1,0,6,1,4,0
2,0,6,1,4,4
3,0,6,1,4,3
4,0,6,1,4,5


In [ ]:
FEATURES = ["hour", "day_of_week", "is_weekend", "time_bucket", "city_enc", "distance_km"]
X = df[FEATURES]
y = df["fare_amount"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training rows: {len(X_train)}")
print(f"Testing rows:  {len(X_test)}")

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("=== Model Performance ===")
print(f"R²:  {r2_score(y_test, y_pred):.3f}")
print(f"MAE: ${mean_absolute_error(y_test, y_pred):.2f}")

print("\n=== Feature Coefficients ===")
for feat, coef in zip(FEATURES, model.coef_):
    print(f"  {feat:<15} {coef:+.4f}")
print(f"  {'intercept':<15} {model.intercept_:+.4f}")

Training rows: 34032
Testing rows:  8508
=== Model Performance ===
R²:  0.750
MAE: $2.49

=== Feature Coefficients ===
  hour            -0.0030
  day_of_week     -0.0002
  is_weekend      -0.0015
  time_bucket     -0.0182
  city_enc        -0.0052
  distance_km     +1.8571
  intercept       +3.0592
